# 01 — GPCRMD Feature Extraction for β2AR Sonification

**Project:** *Listening to GPCR Dynamics: Sonification of β2AR Molecular Dynamics Trajectories*

This notebook is the first of three. It takes three GPCRMD-derived
molecular dynamics trajectories of the β2 adrenergic receptor and
extracts activation-related geometric features that will later be
mapped to musical parameters.

### Systems analyzed

| Label                 | GPCRMD Dynamic ID | Description                                        |
|-----------------------|-------------------|----------------------------------------------------|
| `inactive`            | 11                | Apo, inactive-state β2AR                           |
| `active`              | 116               | Apo, active-state β2AR (G-protein-coupled-like)    |
| `active_ligand_bound` | 117               | Active-state β2AR with orthosteric agonist bound   |

### Features extracted per frame

1. **RMSD_Ca_A** — global Cα RMSD relative to the reference frame, after rigid-body alignment of the whole trajectory.
2. **TM3_TM6_distance_A** — distance between the intracellular Cα centers of TM3 (DRY-motif region) and TM6 (residues at the cytoplasmic end). The hallmark activation marker.
3. **NPxxY_RMSD_A** — RMSD of the NPxxY motif (TM7) relative to the reference frame, computed *after* whole-protein alignment so the value reflects internal motif rearrangement rather than rigid-body drift.
4. **DRY_ionic_lock_A** — distance between the DRY-motif arginine (TM3) and the conserved acidic residue at TM6 (the classic "ionic lock"). Broken upon activation.
5. **Ligand_min_distance_A** — minimum heavy-atom distance between the bound ligand and the receptor (NaN for apo systems).
6. **Ligand_contact_count** — number of receptor heavy atoms within 4.5 Å of any ligand atom (NaN for apo systems).

### Pipeline philosophy

- Each trajectory is **rigid-body-aligned once** on Cα atoms using `MDAnalysis.analysis.align.AlignTraj`. All subsequent RMSDs are computed **without further superposition**, so they reflect genuine internal conformational change.
- Features are saved as per-state CSVs and a unified `all_features.csv` that downstream notebooks consume.
- Residue-number selections are written as soft heuristics (residue-range fallbacks) so that minor numbering differences between GPCRMD topologies do not break the pipeline.


## 1. Mount Google Drive

The pipeline reads trajectory data from and writes all results to
`MyDrive/GPCR_Sonification/`. The expected layout is:

```text
MyDrive/GPCR_Sonification/
├── data/
│   ├── raw/
│   │   ├── inactive/{topology.pdb, trajectory.xtc}
│   │   ├── active/{topology.pdb, trajectory.xtc}
│   │   └── active_ligand_bound/{topology.pdb, trajectory.xtc}
│   └── processed/
└── outputs/
    ├── audio/
    ├── figures/
    └── tables/
```

All subfolders are auto-created if missing, so a fresh Drive only needs the raw trajectory files in place.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 2. Install dependencies

MDAnalysis is the core trajectory-handling library. tqdm gives a progress bar over the trajectory iterator. matplotlib is used for QC plots; the publication-style figure rendering itself happens in notebook 03.


In [2]:
# Install ONLY what's missing on a fresh Colab.
# Colab already ships numpy 2.x, pandas, matplotlib, scipy — do NOT
# re-install those, that's what was breaking the kernel.
# If you are RE-RUNNING this cell after a previous broken install,
# you must FIRST restart the runtime: Runtime -> Restart session.
!pip -q install "MDAnalysis>=2.8.0" tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 3.0 MB/s eta 0:00:00


## 3. Imports, paths, and folder creation

This cell defines the canonical project paths and creates every
subfolder if it does not already exist. It is safe to re-run.


In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import MDAnalysis as mda
from MDAnalysis.analysis import align, rms
from scipy.spatial.distance import cdist

# Canonical project layout
PROJECT_DIR = Path('/content/drive/MyDrive/GPCR_Sonification')
RAW_DIR = PROJECT_DIR / 'data' / 'raw'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
FIG_DIR = PROJECT_DIR / 'outputs' / 'figures'
AUDIO_DIR = PROJECT_DIR / 'outputs' / 'audio'
TABLE_DIR = PROJECT_DIR / 'outputs' / 'tables'

for d in [RAW_DIR, PROCESSED_DIR, FIG_DIR, AUDIO_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Project directory:', PROJECT_DIR)
print('Subfolders ready under outputs/ and data/.')


Project directory: /content/drive/MyDrive/GPCR_Sonification
Subfolders ready under outputs/ and data/.


## 4. System configuration

Each entry below maps a logical state name to its topology and
trajectory files inside `data/raw/`. The GPCRMD Dynamic ID is recorded
for citation purposes.

A `has_ligand` flag is used downstream to decide whether ligand-related
features are meaningful for that system.


In [4]:
# Accepted file extensions (case-insensitive). The first matching file
# in each state subfolder is used.
TOPOLOGY_EXTENSIONS  = ['.pdb', '.prmtop', '.parm7', '.psf', '.gro', '.top', '.tpr']
TRAJECTORY_EXTENSIONS = ['.xtc', '.trr', '.dcd', '.nc', '.netcdf', '.crd']

def find_input_files(state_dir):
    '''Return (topology_path, trajectory_path) by extension.
    Robust against Google Drive FUSE quirks: we don't rely on is_file()
    (which can return False on Drive even when the file is really there);
    we only exclude directories and check the file extension. Verbose:
    prints every file present in the directory so any future discovery
    failure is easy to diagnose.'''
    if not state_dir.exists():
        print(f'    [diag] directory does NOT exist: {state_dir}')
        return None, None
    try:
        entries = sorted(state_dir.iterdir())
    except Exception as e:
        print(f'    [diag] cannot list {state_dir}: {e}')
        return None, None
    if not entries:
        print(f'    [diag] directory is empty')
        return None, None

    # Print everything in the folder for debugging
    print(f'    [diag] folder contents ({len(entries)} entries):')
    for f in entries:
        try:
            size_mb = f.stat().st_size / 1024 / 1024
            print(f'      - {f.name}   ({size_mb:.1f} MB)')
        except Exception:
            print(f'      - {f.name}   (stat unavailable)')

    topology, trajectory = None, None
    for f in entries:
        try:
            if f.is_dir():
                continue
        except Exception:
            # If we can't even decide if it's a directory, skip safely
            continue
        ext = f.suffix.lower()
        if ext in TOPOLOGY_EXTENSIONS and topology is None:
            topology = f
        elif ext in TRAJECTORY_EXTENSIONS and trajectory is None:
            trajectory = f
    return topology, trajectory

# Logical state metadata. File paths are filled in below by find_input_files.
state_meta = {
    'inactive':            {'gpcrmd_id': 11,  'has_ligand': False},
    'active':              {'gpcrmd_id': 116, 'has_ligand': False},
    'active_ligand_bound': {'gpcrmd_id': 117, 'has_ligand': True},
}

# Sanity-check that Drive is actually mounted before we walk subfolders
if not Path('/content/drive/MyDrive').exists():
    print('!!! /content/drive/MyDrive does NOT exist — Drive is not mounted.')
    print('    Re-run the drive.mount cell at the top of the notebook.')
elif not RAW_DIR.exists():
    print(f'!!! {RAW_DIR} does NOT exist.')
    print('    Check that the folder structure exists in your Drive.')

SYSTEMS = {}
print(f'\nScanning {RAW_DIR}/ for trajectory inputs ...')
for state, meta in state_meta.items():
    state_dir = RAW_DIR / state
    print(f'\n  [{state}]  (GPCRMD ID {meta["gpcrmd_id"]})')
    print(f'    dir: {state_dir}')
    topology, trajectory = find_input_files(state_dir)
    SYSTEMS[state] = {
        'gpcrmd_id':  meta['gpcrmd_id'],
        'has_ligand': meta['has_ligand'],
        'topology':   topology,
        'trajectory': trajectory,
    }
    print(f'    -> topology   : {topology.name if topology else "[not picked]"}')
    print(f'    -> trajectory : {trajectory.name if trajectory else "[not picked]"}')

# Frame stride. Increase for a quick test run; 1 = every frame.
FRAME_STRIDE = 5

# Ligand-contact cutoff (Å). 4.5 Å is a generous heavy-atom contact threshold.
LIGAND_CONTACT_CUTOFF = 4.5

print(f'\nConfigured {len(SYSTEMS)} systems with frame stride {FRAME_STRIDE}')



Scanning /content/drive/MyDrive/GPCR_Sonification/data/raw/ for trajectory inputs ...

  [inactive]  (GPCRMD ID 11)
    dir: /content/drive/MyDrive/GPCR_Sonification/data/raw/inactive
    [diag] folder contents (2 entries):
      - topology.pdb   (6.4 MB)
      - trajectory.xtc   (733.6 MB)
    -> topology   : topology.pdb
    -> trajectory : trajectory.xtc

  [active]  (GPCRMD ID 116)
    dir: /content/drive/MyDrive/GPCR_Sonification/data/raw/active
    [diag] folder contents (2 entries):
      - topology.pdb   (6.9 MB)
      - trajectory.xtc   (792.0 MB)
    -> topology   : topology.pdb
    -> trajectory : trajectory.xtc

  [active_ligand_bound]  (GPCRMD ID 117)
    dir: /content/drive/MyDrive/GPCR_Sonification/data/raw/active_ligand_bound
    [diag] folder contents (2 entries):
      - topology.pdb   (6.9 MB)
      - trajectory.xtc   (790.1 MB)
    -> topology   : topology.pdb
    -> trajectory : trajectory.xtc

Configured 3 systems with frame stride 5


## 5. Atom selections

These residue ranges target activation-related regions in β2AR. They
are intentionally written as **ranges** rather than single residues so
that small numbering offsets between GPCRMD topologies do not break the
pipeline. After mounting your real data, run the validation cell below
to confirm that each selection yields a non-zero number of atoms; if
not, edit the ranges to match your topology.

For β2AR the standard activation-related residues are roughly:

- TM3 intracellular end (DRY motif): R131(3.50) → residues ~128–135
- TM6 intracellular end:             E268(6.30) / L272(6.34) → residues ~265–275
- NPxxY motif (TM7):                 N322–Y326 → residues ~318–328
- Ionic lock pair:                   R131 (TM3) ↔ E268 (TM6)

The DRY ionic-lock pair is defined separately as a precise two-residue
selection because the lock distance is sensitive to which residues are picked.


In [8]:
# Atom selections used by the feature-extraction pipeline.
#
# Residue ranges verified by direct inspection of all three GPCRMD
# topology PDBs (identical across them):
#
#   TM3 helix:   103 - 136
#   TM6 helix:   262 - 298
#   NPxxY motif: 322 - 326   (N322-P323-L324-I325-Y326)
#   DRY motif:   130 - 132   (D130-R131-Y132)
#
# For the TM3-TM6 "activation distance" only the intracellular ends
# of each helix are used (TM3 IC = the DRY end; TM6 IC = the E6.30
# end), which is the standard β2AR activation marker in the literature.

# Residue names to EXCLUDE when selecting the ligand. Covers BOTH
# CHARMM and Amber/GROMACS naming conventions for ions, plus POPC
# (the lipid used in these GPCRMD systems) and CYSP (palmitoylated
# cysteine, a post-translational modification of β2AR Cys341 that
# MDAnalysis does not recognise as part of the standard 'protein'
# keyword).
NON_LIGAND_RESNAMES = (
    'POPC POPE POPS POPG CHL1 CHOL CHS '          # lipids / sterols
    'TIP3 TIP3P SOL WAT HOH SPC '                  # waters
    'NA CL K MG CA ZN LI RB '                      # ions (Amber/GROMACS)
    'SOD CLA POT CAL MG2 ZN2 LIT RUB '             # ions (CHARMM)
    'CYSP CYP CYS_PAL'                              # palmitoylated cysteine PTM
)

SELECTIONS = {
    # Whole-protein Cα for alignment and global RMSD
    'protein_ca':    'protein and name CA',

    # Full TM3 and TM6 helices (kept for reference / sanity-check plots)
    'tm3_full_ca':   'protein and name CA and resid 103-136',
    'tm6_full_ca':   'protein and name CA and resid 262-298',

    # TM3 intracellular end — last 7 residues of TM3 (cytoplasmic side,
    # contains the DRY motif). Used in TM3-TM6 distance computation.
    'tm3_ic_ca':     'protein and name CA and resid 130-136',

    # TM6 intracellular end — first 11 residues of TM6 (cytoplasmic side,
    # contains E6.30 = E268 and L6.34 = L272). Used in TM3-TM6 distance.
    'tm6_ic_ca':     'protein and name CA and resid 262-272',

    # NPxxY motif — EXACTLY 5 residues (N-P-x-x-Y)
    'npxxy_ca':      'protein and name CA and resid 322-326',

    # DRY motif — EXACTLY 3 residues (D-R-Y)
    'dry_motif_ca':  'protein and name CA and resid 130-132',

    # Individual Cα atoms for the precise ionic-lock distance
    'dry_R3.50_ca':  'protein and name CA and resid 131',
    'tm6_E6.30_ca':  'protein and name CA and resid 268',

    # Ligand: everything that isn't protein, lipid, water, ion, or PTM
    'ligand':        f'not protein and not resname {NON_LIGAND_RESNAMES}',

    # Protein heavy atoms (used for ligand-receptor contact counts)
    'protein_heavy': 'protein and not name H*',
}

print('Selection definitions:')
for k, v in SELECTIONS.items():
    print(f'  {k:15s}  ->  {v}')

Selection definitions:
  protein_ca       ->  protein and name CA
  tm3_full_ca      ->  protein and name CA and resid 103-136
  tm6_full_ca      ->  protein and name CA and resid 262-298
  tm3_ic_ca        ->  protein and name CA and resid 130-136
  tm6_ic_ca        ->  protein and name CA and resid 262-272
  npxxy_ca         ->  protein and name CA and resid 322-326
  dry_motif_ca     ->  protein and name CA and resid 130-132
  dry_R3.50_ca     ->  protein and name CA and resid 131
  tm6_E6.30_ca     ->  protein and name CA and resid 268
  ligand           ->  not protein and not resname POPC POPE POPS POPG CHL1 CHOL CHS TIP3 TIP3P SOL WAT HOH SPC NA CL K MG CA ZN LI RB SOD CLA POT CAL MG2 ZN2 LIT RUB CYSP CYP CYS_PAL
  protein_heavy    ->  protein and not name H*


## 6. Topology / selection validation

Loads each system once and reports how many atoms each selection
yields. **If any of the protein-related selections come back as zero
atoms, the residue ranges in the previous cell need to be adjusted to
match the actual topology numbering** (GPCRMD topologies sometimes
include construct-specific offsets, signal peptides, or T4-lysozyme
fusions that shift residue indices).


In [9]:
from collections import Counter

def report_selections(universe, label):
    '''Validate selections on a Universe. For small selections (<=12
    atoms) also prints the residue names + numbers so the user can
    visually verify that NPxxY really hit ASN-PRO-x-x-TYR, etc.
    Also prints the composition of the 'ligand' selection so we can
    identify any stray non-protein residues that need to be excluded.'''
    print(f'\n=== {label} ===')
    protein = universe.select_atoms('protein')
    if len(protein) > 0:
        resids = protein.residues.resids
        print(f'  protein:    {len(protein.residues)} residues, '
              f'resid {resids.min()}-{resids.max()}')
    print(f'  trajectory: {len(universe.trajectory)} frames')

    for name, sel in SELECTIONS.items():
        ag = universe.select_atoms(sel)
        flag = 'OK' if len(ag) > 0 else '!!'
        line = f'  [{flag}] {name:15s}  atoms={len(ag):5d}'
        # For small selections, append the actual residue identities so
        # the user can confirm we picked the right residues.
        if 0 < len(ag) <= 12:
            tags = '-'.join(f'{r.resname}{r.resid}' for r in ag.residues)
            line += f'   {tags}'
        print(line)

    # Detailed composition of the 'ligand' selection.
    # In an apo system this MUST be empty; any residue names listed
    # here are what we need to add to the exclusion list (or recognise
    # as a real bound ligand in the holo system).
    ligand_ag = universe.select_atoms(SELECTIONS['ligand'])
    if len(ligand_ag) > 0:
        rn_counts = Counter()
        for r in ligand_ag.residues:
            rn_counts[r.resname] += 1
        print(f'  >>> ligand selection composition (resname  #residues  #atoms):')
        for resname, n_residues in rn_counts.most_common():
            n_atoms = len(ligand_ag.select_atoms(f'resname {resname}'))
            print(f'        {resname:8s}  {n_residues:4d} residues, {n_atoms:5d} atoms')

for state, cfg in SYSTEMS.items():
    if cfg['topology'] is None or cfg['trajectory'] is None:
        print(f'\n[SKIP] {state}: missing topology or trajectory')
        continue
    u = mda.Universe(str(cfg['topology']), str(cfg['trajectory']))
    report_selections(u, f"{state} (GPCRMD ID {cfg['gpcrmd_id']})")


=== inactive (GPCRMD ID 11) ===
  protein:    302 residues, resid 26-342
  trajectory: 2500 frames
  [OK] protein_ca       atoms=  302
  [OK] tm3_full_ca      atoms=   34
  [OK] tm6_full_ca      atoms=   37
  [OK] tm3_ic_ca        atoms=    7   ASP130-ARG131-TYR132-PHE133-ALA134-ILE135-THR136
  [OK] tm6_ic_ca        atoms=   11   SER262-LYS263-PHE264-CYS265-LEU266-LYS267-GLU268-HSD269-LYS270-ALA271-LEU272
  [OK] npxxy_ca         atoms=    5   ASN322-PRO323-LEU324-ILE325-TYR326
  [OK] dry_motif_ca     atoms=    3   ASP130-ARG131-TYR132
  [OK] dry_R3.50_ca     atoms=    1   ARG131
  [OK] tm6_E6.30_ca     atoms=    1   GLU268
  [!!] ligand           atoms=    0
  [OK] protein_heavy    atoms= 2450

=== active (GPCRMD ID 116) ===
  protein:    303 residues, resid 26-342
  trajectory: 2500 frames
  [OK] protein_ca       atoms=  303
  [OK] tm3_full_ca      atoms=   34
  [OK] tm6_full_ca      atoms=   37
  [OK] tm3_ic_ca        atoms=    7   ASP130-ARG131-TYR132-PHE133-ALA134-ILE135-THR136
  

## 7. Feature extraction

### Methodology

For each system the pipeline does the following:

1. Loads topology + trajectory.
2. **Pre-aligns the whole trajectory in memory to its first frame** on Cα atoms (`AlignTraj`). This removes rigid-body translation/rotation.
3. Captures reference Cα positions and reference NPxxY positions from frame 0 of the *aligned* trajectory.
4. Iterates over frames at the configured stride and, for each:
   - Computes Cα RMSD vs. the reference (no further superposition).
   - Computes NPxxY motif RMSD vs. the reference (no further superposition — now meaningful because the bundle is already aligned).
   - Computes the TM3-TM6 intracellular distance as the distance between Cα centers of the two selections.
   - Computes the DRY ionic-lock distance (R131-Cα to E268-Cα; falls back to NaN if either residue is missing).
   - If `has_ligand` is True, computes the minimum heavy-atom ligand-receptor distance and the contact count within `LIGAND_CONTACT_CUTOFF`. For apo systems these are left as NaN.

The output is one CSV per state plus a unified `all_features.csv`.


In [10]:
def safe_center(ag):
    if len(ag) == 0:
        return np.array([np.nan, np.nan, np.nan])
    return ag.center_of_geometry()

def pair_distance(ag1, ag2):
    if len(ag1) == 0 or len(ag2) == 0:
        return np.nan
    return float(np.linalg.norm(ag1.positions[0] - ag2.positions[0]))

def min_distance_between(ag1, ag2):
    if len(ag1) == 0 or len(ag2) == 0:
        return np.nan
    return float(cdist(ag1.positions, ag2.positions).min())

def contact_count(ag1, ag2, cutoff):
    if len(ag1) == 0 or len(ag2) == 0:
        return np.nan
    return int((cdist(ag1.positions, ag2.positions) < cutoff).sum())


In [11]:
def extract_features_for_system(state_name, cfg, selections,
                                stride=5, contact_cutoff=4.5):
    '''Extract the activation-related feature panel for one trajectory.'''
    print(f'\n--- Extracting {state_name} ---')

    u = mda.Universe(str(cfg['topology']), str(cfg['trajectory']))
    n_total = len(u.trajectory)
    print(f'  total frames: {n_total}')

    # Step 1: rigid-body-align the whole trajectory once on Cα atoms.
    print('  pre-aligning trajectory on Cα atoms ...')
    align.AlignTraj(
        mobile=u, reference=u,
        select=selections['protein_ca'],
        in_memory=True,
        verbose=False,
    ).run()

    # Atom groups (resolve after AlignTraj has populated in-memory coords)
    protein_ca    = u.select_atoms(selections['protein_ca'])
    tm3           = u.select_atoms(selections['tm3_ic_ca'])
    tm6           = u.select_atoms(selections['tm6_ic_ca'])
    npxxy         = u.select_atoms(selections['npxxy_ca'])
    dry_arg       = u.select_atoms(selections['dry_R3.50_ca'])
    tm6_glu       = u.select_atoms(selections['tm6_E6.30_ca'])
    ligand        = u.select_atoms(selections['ligand']) if cfg['has_ligand'] else None
    protein_heavy = u.select_atoms(selections['protein_heavy'])

    if len(protein_ca) == 0:
        raise ValueError(f'{state_name}: protein_ca selection has 0 atoms.')

    # Reference positions from frame 0 of the aligned trajectory
    u.trajectory[0]
    ref_ca_pos    = protein_ca.positions.copy()
    ref_npxxy_pos = npxxy.positions.copy() if len(npxxy) > 0 else None

    rows = []
    for ts in tqdm(u.trajectory[::stride], desc='  frames'):
        frame   = int(ts.frame)
        time_ps = float(getattr(ts, 'time', frame))

        # 1) Global Cα RMSD (bundle pre-aligned, so no further superposition)
        ca_rmsd = rms.rmsd(protein_ca.positions, ref_ca_pos,
                           center=False, superposition=False)

        # 2) NPxxY motif RMSD (now meaningful: bundle aligned, so this
        # reflects internal motif rearrangement only)
        if ref_npxxy_pos is not None and len(npxxy) > 0:
            npxxy_rmsd = rms.rmsd(npxxy.positions, ref_npxxy_pos,
                                  center=False, superposition=False)
        else:
            npxxy_rmsd = np.nan

        # 3) TM3-TM6 intracellular-end Cα-centre distance
        c3 = safe_center(tm3)
        c6 = safe_center(tm6)
        tm3_tm6_dist = (float(np.linalg.norm(c3 - c6))
                        if not (np.any(np.isnan(c3)) or np.any(np.isnan(c6)))
                        else np.nan)

        # 4) DRY ionic-lock distance (R3.50 Cα ↔ E6.30 Cα)
        ionic_lock = pair_distance(dry_arg, tm6_glu)

        # 5/6) Ligand observables (only when ligand present)
        if cfg['has_ligand'] and ligand is not None and len(ligand) > 0:
            lig_min = min_distance_between(ligand, protein_heavy)
            lig_cnt = contact_count(ligand, protein_heavy, cutoff=contact_cutoff)
        else:
            lig_min, lig_cnt = np.nan, np.nan

        rows.append({
            'state':                 state_name,
            'gpcrmd_id':             cfg['gpcrmd_id'],
            'frame':                 frame,
            'time_ps':               time_ps,
            'RMSD_Ca_A':             ca_rmsd,
            'TM3_TM6_distance_A':    tm3_tm6_dist,
            'NPxxY_RMSD_A':          npxxy_rmsd,
            'DRY_ionic_lock_A':      ionic_lock,
            'Ligand_min_distance_A': lig_min,
            'Ligand_contact_count':  lig_cnt,
        })

    df = pd.DataFrame(rows)
    print(f'  extracted {len(df)} frames')
    return df


# Run extraction for all configured systems
all_features = []
for state, cfg in SYSTEMS.items():
    if cfg['topology'] is None or cfg['trajectory'] is None:
        print(f'[SKIP] {state}: missing files')
        continue
    df_state = extract_features_for_system(
        state_name=state,
        cfg=cfg,
        selections=SELECTIONS,
        stride=FRAME_STRIDE,
        contact_cutoff=LIGAND_CONTACT_CUTOFF,
    )
    out_csv = PROCESSED_DIR / f'{state}_features.csv'
    df_state.to_csv(out_csv, index=False)
    print('  saved:', out_csv)
    all_features.append(df_state)

if all_features:
    df_all = pd.concat(all_features, ignore_index=True)
    df_all.to_csv(PROCESSED_DIR / 'all_features.csv', index=False)
    print('\nUnified table saved to', PROCESSED_DIR / 'all_features.csv')
    print('Per-state frame counts:')
    print(df_all['state'].value_counts())
    display(df_all.head())
else:
    print('\nNo features extracted.')



--- Extracting inactive ---
  total frames: 2500
  pre-aligning trajectory on Cα atoms ...


  frames:   0%|          | 0/500 [00:00<?, ?it/s]

  extracted 500 frames
  saved: /content/drive/MyDrive/GPCR_Sonification/data/processed/inactive_features.csv

--- Extracting active ---
  total frames: 2500
  pre-aligning trajectory on Cα atoms ...


  frames:   0%|          | 0/500 [00:00<?, ?it/s]

  extracted 500 frames
  saved: /content/drive/MyDrive/GPCR_Sonification/data/processed/active_features.csv

--- Extracting active_ligand_bound ---
  total frames: 2500
  pre-aligning trajectory on Cα atoms ...


  frames:   0%|          | 0/500 [00:00<?, ?it/s]

  extracted 500 frames
  saved: /content/drive/MyDrive/GPCR_Sonification/data/processed/active_ligand_bound_features.csv

Unified table saved to /content/drive/MyDrive/GPCR_Sonification/data/processed/all_features.csv
Per-state frame counts:
state
inactive               500
active                 500
active_ligand_bound    500
Name: count, dtype: int64


,state,gpcrmd_id,frame,time_ps,RMSD_Ca_A,TM3_TM6_distance_A,NPxxY_RMSD_A,DRY_ionic_lock_A,Ligand_min_distance_A,Ligand_contact_count
0,inactive,11,0,0.0,0.000000,13.846714,0.000000,10.681729,NaN,NaN
1,inactive,11,5,1000000.0,1.535102,14.144362,1.598755,10.152783,NaN,NaN
2,inactive,11,10,2000000.0,1.379962,13.398729,1.422459,9.590470,NaN,NaN
3,inactive,11,15,3000000.0,1.353491,13.407813,1.183302,9.586854,NaN,NaN
4,inactive,11,20,4000000.0,2.124453,14.159109,1.386260,10.422890,NaN,NaN


In [13]:
df = pd.read_csv(PROCESSED_DIR / 'all_features.csv')
print(df.groupby('state')[['Ligand_min_distance_A', 'Ligand_contact_count']].describe())
print('\nactive_ligand_bound ilk 5 satır:')
print(df[df['state']=='active_ligand_bound'].head())

                    Ligand_min_distance_A                                \
                                    count      mean       std       min   
state                                                                     
active                                0.0       NaN       NaN       NaN   
active_ligand_bound                 500.0  1.805607  0.169431  1.448413   
inactive                              0.0       NaN       NaN       NaN   

                                                   Ligand_contact_count  \
                          25%   50%      75%   max                count   
state                                                                     
active                    NaN   NaN      NaN   NaN                  0.0   
active_ligand_bound  1.696054  1.78  1.86804  2.65                500.0   
inactive                  NaN   NaN      NaN   NaN                  0.0   

                                                                            
                     

## 8. Quality-control plots

Quick time-series plots that show each feature side-by-side across all three states. These are diagnostic plots — the polished, publication-style versions are produced by notebook 03. Here we only check that:

- features are non-trivially time-varying;
- the three states show distinguishable distributions;
- there are no large gaps or numerical artefacts.


In [ ]:
import matplotlib as mpl

# ---------- Publication-style matplotlib (Nature-like) ----------
plt.rcParams.update({
    'font.family':         'sans-serif',
    'font.sans-serif':     ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size':           9,
    'axes.titlesize':      10,
    'axes.titleweight':    'bold',
    'axes.labelsize':      9,
    'axes.linewidth':      0.8,
    'axes.spines.top':     False,
    'axes.spines.right':   False,
    'xtick.labelsize':     8,
    'ytick.labelsize':     8,
    'xtick.direction':     'out',
    'ytick.direction':     'out',
    'xtick.major.size':    3.0,
    'ytick.major.size':    3.0,
    'xtick.major.width':   0.7,
    'ytick.major.width':   0.7,
    'legend.fontsize':     8,
    'legend.frameon':      False,
    'legend.handlelength': 1.4,
    'figure.dpi':          110,
    'savefig.dpi':         600,
    'savefig.bbox':        'tight',
    'savefig.pad_inches':  0.05,
    'pdf.fonttype':        42,    # editable text in Illustrator/Inkscape
    'ps.fonttype':         42,
})

# Colorblind-safe Okabe-Ito palette (matches downstream notebooks)
STATE_COLORS = {
    'inactive':            '#0072B2',  # blue
    'active':              '#E69F00',  # orange
    'active_ligand_bound': '#009E73',  # green
}

# Pretty display names used in legends
STATE_DISPLAY = {
    'inactive':            'inactive (apo)',
    'active':              'active (apo)',
    'active_ligand_bound': 'active + ligand',
}

STATE_ORDER = ['inactive', 'active', 'active_ligand_bound']

# Feature metadata: pretty label + Y-axis unit
FEATURE_INFO = {
    'RMSD_Ca_A':             ('RMSD (Cα)',                               'Å'),
    'TM3_TM6_distance_A':    ('TM3–TM6 intracellular distance',          'Å'),
    'NPxxY_RMSD_A':          ('NPxxY motif RMSD',                        'Å'),
    'DRY_ionic_lock_A':      ('DRY ionic-lock distance (R3.50–E6.30)',   'Å'),
    'Ligand_min_distance_A': ('Ligand–receptor minimum distance',        'Å'),
    'Ligand_contact_count':  ('Ligand–receptor contacts',                'atoms'),
}

QC_FEATURES = [
    'RMSD_Ca_A',
    'TM3_TM6_distance_A',
    'NPxxY_RMSD_A',
    'DRY_ionic_lock_A',
    'Ligand_min_distance_A',
    'Ligand_contact_count',
]

if (PROCESSED_DIR / 'all_features.csv').exists():
    df_all = pd.read_csv(PROCESSED_DIR / 'all_features.csv')

    for feature in QC_FEATURES:
        if feature not in df_all.columns or df_all[feature].dropna().empty:
            print(f'  skipping {feature}: no data')
            continue

        label, unit = FEATURE_INFO.get(feature, (feature, ''))

        # constrained_layout reserves space for the external legend so
        # it never overlaps the data, gets clipped on save, or fights
        # with tight_layout() warnings in Colab.
        fig, ax = plt.subplots(figsize=(8.5, 3.2), constrained_layout=True)

        for state in STATE_ORDER:
            g = df_all[df_all['state'] == state]
            if g[feature].dropna().empty:
                continue
            ax.plot(g['frame'], g[feature],
                    color=STATE_COLORS.get(state, 'gray'),
                    linewidth=1.0, alpha=0.9,
                    label=STATE_DISPLAY.get(state, state))

        ax.set_xlabel('Frame')
        ax.set_ylabel(f'{label}  ({unit})' if unit else label)
        ax.set_title(label, loc='left', pad=8, fontweight='bold')

        # Legend OUTSIDE the axes to the right — never overlaps data
        ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0),
                  borderaxespad=0)

        # Subtle horizontal gridlines (Nature-style readability aid)
        ax.grid(True, axis='y', alpha=0.25, linewidth=0.5, linestyle='-')
        ax.set_axisbelow(True)

        # Small left/right padding inside the plot area
        ax.margins(x=0.005)

        # Save in PNG + PDF + SVG at 600 dpi
        base = FIG_DIR / f'QC_{feature}'
        for ext in ('png', 'pdf', 'svg'):
            fig.savefig(f'{base}.{ext}')
        plt.show()
        plt.close(fig)
        print(f'  saved: {base.name}.[png,pdf,svg]')
else:
    print(f'  {PROCESSED_DIR / "all_features.csv"} not found — run cell 7 first.')

## Output of this notebook

Files written to Drive:

```text
data/processed/inactive_features.csv
data/processed/active_features.csv
data/processed/active_ligand_bound_features.csv
data/processed/all_features.csv
outputs/figures/QC_*.png
```

Proceed to **notebook 02** to map these features to MIDI / WAV audio,
and **notebook 03** to produce the publication-quality figures and
tables.
